# 4.3 One distribution, or several groups — penguins

[04.2](04.2-long-tail.ipynb) fitted a family to taxi fares and the fitter agreed with the
mechanism: one process, one long tail, one family that explains the expensive rides. This
notebook is about the case where the fitter *cannot* agree with anything — not because the
data is strange, but because it is not one population. The question it answers: **when a fit
fails, is the family wrong, or is the grouping?** It assumes 04.2's tools — `DistributionFitter`,
`fit_table` with its two winners, `PlotFits`, and a qq-plot read for the tail — and 04.1's
mechanisms, the central limit theorem in particular: a measurement that is the sum of many
small contributions should be normal.

The data is the penguins showcase from [02.2](../lesson2/02.2-comparing_categories.ipynb):
three species, two sexes, four measurements per bird. Lesson 2 compared their averages. Here
the variable is **flipper length**, and the first move is the wrong one on purpose: pool every
bird and fit.

In [ ]:
import numpy as np
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult, fit_table
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    DistPlot,
    ECDFPlot,
    FitPlotSettings,
    HistogramPlot,
    PlotFits,
    PlotSettings,
    QQPlot,
)

from wa_analyzer.data import load_showcase

penguins = load_showcase("penguins").dropna(subset=["flipper_length_mm", "sex"]).reset_index(drop=True)
flipper = penguins["flipper_length_mm"].to_numpy().astype(float)
print(f"{len(penguins)} birds with a flipper measurement and a recorded sex")
print(f"mean: {flipper.mean():.1f} mm   median: {np.median(flipper):.1f} mm")

## 4.3.1 Pool everything and fit

A flipper grows by many small increments over a bird's development, and a sum of many
contributions is the mechanism behind the normal (04.1 §4.1.2). So the hypothesis is `norm`,
and the pooled fit is the check. Nothing in the printout warns you: mean and median are close,
which is what a symmetric family predicts. The histogram is a different matter.

In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
flipper_fits = fitter.fit(flipper, discrete=False)

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="flipper length (mm)", ylabel="density",
                            title="All 333 birds: the top three fits")).plot(
    data=flipper, fit_results=flipper_fits, fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

In [ ]:
fit_table(flipper_fits)[["distribution", "log_likelihood", "ks_stat", "ks_pvalue", "best_likelihood", "best_ks"]]

Two humps: a large one around 190 mm, a smaller one around 215 mm, and a trough between them
where hardly any bird sits. No family in the registry has two humps, so every curve is a
compromise. The log-likelihood winner, `beta`, is the most honest compromise available — one
wide, flat hump over both peaks — and the KS winner is a different family again. In 04.2 the
two winners agreed; here they disagree, *and the p-value column rejects every family in the
table*. That combination is the most informative outcome `fit_table` can produce, and it does
not mean the fitter failed. Maximum likelihood found the best single curve through these
numbers; the numbers were never produced by a single curve.

The qq-plot shows where the compromise breaks, and in a way a histogram cannot: it shows
*which* values the family has no room for.

In [ ]:
best = next(f for f in flipper_fits if isinstance(f, FitResult) and f.best_likelihood)
norm_fit = fitter.fit_distribution("norm", flipper)

qq = PlotSettings(
    figsize=(10, 4.5),  # ty: ignore[invalid-argument-type]
    title="Pooled flipper length: the fitted family has no room for a gap",
    subplot_titles=[f"flipper vs. fitted {best.distribution}", "flipper vs. fitted normal"],
    xlabel="theoretical quantile",
    ylabel="flipper length (mm)",
    max_cols=2,
)
host = QQPlot(qq)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(QQPlot(qq), axes[0], data=flipper, distribution=best.frozen_dist)
_ = host.plot_on_axes(QQPlot(qq), axes[1], data=flipper, distribution=norm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]

Both panels have the same **kink**. Through the crowd between about 190 and 205 mm the points
fall further and further below the line — there are more birds here than the family expects —
and then, just past 205 mm, they jump almost vertically across it: a stretch of theoretical
quantiles the data has no birds for. That is the trough in the histogram, seen from the side. A single family, whatever its parameters, has no way to produce a gap in the middle of
its own range. The fit is not lying: the beta really is the best single curve through this
data. The *family* is wrong for the *population*, and it is wrong because there is no single
population here. Lesson 2 already said what the groups are.

## 4.3.2 Split by species: three shapes

The same variable, once per species, on one bin-free axis — `ECDFPlot` takes a second sample
as `compare`, and a third can be layered on with `plot_on`.

In [ ]:
by_species = {name: group["flipper_length_mm"].to_numpy().astype(float) for name, group in penguins.groupby("species")}

species_ecdf = PlotSettings(figsize=(9, 4), title="Flipper length per species: three curves, not one",
                            xlabel="flipper length (mm)", ylabel="share of birds at or below")
ecdf = ECDFPlot(species_ecdf)
fig, ax = ecdf.plot(data=by_species["Adelie"], compare=by_species["Chinstrap"], label="Adelie", compare_label="Chinstrap")
_ = ecdf.plot_on(ECDFPlot(species_ecdf), data=by_species["Gentoo"], label="Gentoo", color="darkgreen")

In [ ]:
def describe_fit(name: str, values: np.ndarray) -> list:
    """Fit every continuous family and report how the normal did against the field."""
    fits = fitter.fit(values, discrete=False)
    table = fit_table(fits)
    top = table["log_likelihood"].max()
    normal = table.loc[table["distribution"] == "norm"].iloc[0]
    within_three = int((table["log_likelihood"] > top - 3).sum())
    rejected = ", ".join(table.loc[table["ks_pvalue"] < 0.05, "distribution"]) or "none"
    print(f"{name:14s} n={len(values):3d}  mean {values.mean():6.1f}  sd {values.std(ddof=1):4.1f}  |  "
          f"norm: KS p={normal['ks_pvalue']:.2f}, {normal['log_likelihood'] - top:+.1f} from the winner;  "
          f"{within_three} families within 3 log-lik points;  rejected: {rejected}")
    return fits


species_hist = PlotSettings(
    figsize=(13, 3.5),  # ty: ignore[invalid-argument-type]
    title="Each species on its own, with its fitted normal",
    subplot_titles=list(by_species),
    xlabel="flipper length (mm)",
    max_cols=3,
    sharex=True,
)
host = HistogramPlot(species_hist)
fig, axes = host.create_figure(n_plots=3)
for ax, (name, values) in zip(axes, by_species.items()):
    describe_fit(name, values)
    host.plot_on_axes(HistogramPlot(species_hist), ax, data=values, bins=15, color="lightgrey")
    host.plot_on_axes(DistPlot(species_hist), ax, distribution=fitter.fit_distribution("norm", values).frozen_dist)  # ty: ignore[unresolved-attribute]
    ax.set_ylabel("")

Three bells. Adelie and Chinstrap overlap, with Chinstrap a few millimetres to the right;
Gentoo sits more than twenty millimetres further along, and the empty stretch between the
Chinstrap curve and the Gentoo curve in the ECDF is the trough in the pooled histogram and the
kink in the qq-plot. Per species the fitter is content: the normal is not rejected for any of
the three, it is the winner or within a few log-likelihood points of it (four, for Gentoo,
against a beta with two extra parameters), and five or six families sit within three points
of the top. That tie is the reading: with a hundred-odd birds the data cannot tell a normal
from a Weibull or a skew-normal, and when the data has not distinguished the families, the
family the mechanism predicts is as good as any. The families that *are* rejected — uniform
and exponential everywhere, gamma for Adelie at the margin — are rejected for the right
reason: the bell is not flat and does not start at a wall.

So the fit that failed in §4.3.1 was not a failure of `norm`. It was a failure of the row: a
"penguin" is not a unit that has a flipper length, a *species* of penguin is.

## 4.3.3 The same move once more: Gentoo by sex

A passing fit is not proof that you have reached a single population. Gentoo alone passed
every check above, and 02.2's grouped bars already showed that the sexes differ in size. Split
once more.

In [ ]:
gentoo = penguins[penguins["species"] == "Gentoo"]
by_sex = {sex: group["flipper_length_mm"].to_numpy().astype(float) for sex, group in gentoo.groupby("sex")}

describe_fit("Gentoo, pooled", by_species["Gentoo"])
for sex, values in by_sex.items():
    describe_fit(f"Gentoo {sex.lower()}", values)

sex_split = PlotSettings(
    figsize=(12, 4),
    title="Gentoo flipper length by sex",
    subplot_titles=["two histograms, two fitted normals", "the same two samples as ECDFs"],
    xlabel="flipper length (mm)",
    max_cols=2,
)
host = HistogramPlot(sex_split)
fig, axes = host.create_figure(n_plots=2)
for (sex, values), colour in zip(by_sex.items(), ["steelblue", "crimson"]):
    host.plot_on_axes(HistogramPlot(sex_split), axes[0], data=values, bins=12, color=colour, alpha=0.4)
    host.plot_on_axes(DistPlot(sex_split), axes[0], distribution=fitter.fit_distribution("norm", values).frozen_dist,  # ty: ignore[unresolved-attribute]
                      color=colour, label=sex.lower())
axes[0].legend()
_ = host.plot_on_axes(ECDFPlot(sex_split), axes[1], data=by_sex["Female"], compare=by_sex["Male"],
                      label="female", compare_label="male")

Two bells, nine millimetres apart, each narrower than the one they were hiding in: the pooled
Gentoo standard deviation of 6.6 mm becomes 3.9 mm for the females and 5.7 mm for the males,
and the normal is not rejected for either (the male histogram is ragged, but that is what
sixty-one birds measured to the millimetre look like). The pooled Gentoo fit passed KS with
p ≈ 0.2 and was a mixture all along. That is not a weakness of the test: two bells less than two standard deviations
apart add up to a shape that is itself very nearly a bell, and no fitter can see through it.
What told you to split was not the fit table. It was knowing the data — 02.2's question,
*what is a row here*, and its answer: one bird, of a species and a sex.

A mixture hides its groups the way [02.3](../lesson2/02.3-simpsons_paradox.ipynb)'s aggregate
hid its subgroups: the pooled summary is arithmetically correct and describes nobody.

## 4.3.4 An outlier is a claim about a population

Reason 2 from [04.1](04.1-families.ipynb): a point is only unusual *relative to a
distribution*. This data has two candidate distributions for every bird — the pooled fit from
§4.3.1, and the normal for its own species and sex — and they give different answers to the
same question. Start with the longest flipper in the set.

In [ ]:
by_group = {
    key: fitter.fit_distribution("norm", group["flipper_length_mm"].to_numpy().astype(float)).frozen_dist  # ty: ignore[unresolved-attribute]
    for key, group in penguins.groupby(["species", "sex"])
}
pooled = best.frozen_dist

longest = penguins.loc[penguins["flipper_length_mm"].idxmax()]
value = float(longest["flipper_length_mm"])
own = by_group[(longest["species"], longest["sex"])]

pooled_tail = 1 - pooled.cdf(value)
own_tail = 1 - own.cdf(value)
print(f"longest flipper: {value:.0f} mm, a {longest['sex'].lower()} {longest['species']}")
print(f"P(flipper >= {value:.0f}) under the pooled {best.distribution}:  {pooled_tail:.4f}  (~1-in-{1 / pooled_tail:,.0f})")
print(f"    ...whose support ends at {pooled.support()[1]:.0f} mm: it puts zero probability on anything longer")
print(f"P(flipper >= {value:.0f}) under the {longest['species']} {longest['sex'].lower()} normal: {own_tail:.3f}  (~1-in-{1 / own_tail:,.0f})")

Under the pooled fit this is a one-in-a-hundred-odd bird, and the fitted beta — a bounded
family — goes further: it puts a hard ceiling a few millimetres above it, a claim that no
penguin can have a flipper longer than that. Under the fit for its own group it is a large
male Gentoo, the kind you meet every twenty or so. Same bird, same measurement; the difference
is which population the question is about.

The disagreement runs the other way too, and that direction is the dangerous one. Score every
bird under both fits — the two-sided tail probability, how much of the distribution lies at
least this far from the centre — and look at where the two answers differ most.

In [ ]:
def tail(dist, x: np.ndarray) -> np.ndarray:
    """Two-sided tail probability: the share of `dist` at least this far from its centre."""
    p = dist.cdf(x)
    return 2 * np.minimum(p, 1 - p)


flagged = penguins[["species", "sex", "flipper_length_mm"]].copy()
flagged["tail_pooled"] = tail(pooled, flipper)
flagged["tail_own_group"] = [
    tail(by_group[(species, sex)], value)
    for species, sex, value in zip(flagged["species"], flagged["sex"], flagged["flipper_length_mm"])
]
threshold = 0.01  # roughly the two-sided "z beyond 2.6" line, but under whichever family is asked

print(f"birds beyond the {threshold} tail: {int((flagged.tail_pooled < threshold).sum())} under the pooled fit, "
      f"{int((flagged.tail_own_group < threshold).sum())} under their own group's fit")
flagged["ratio"] = flagged["tail_pooled"] / flagged["tail_own_group"]
pd.concat([
    flagged.nlargest(3, "ratio").assign(verdict="ordinary pooled, unusual for its group"),
    flagged.nsmallest(3, "ratio").assign(verdict="unusual pooled, less so for its group"),
]).drop(columns="ratio").round(4)

The top rows are the ones to worry about. An Adelie female at 202 mm sits almost exactly at
the pooled median — the tail probability says nothing could be more ordinary — and she is two and
a half standard deviations long for an Adelie female, a one-in-a-hundred bird in her own group. The pooled fit cannot see her, because at 202 mm she is standing in the
crowd of another species. The bottom rows are the reverse: the shortest and longest flippers
look rarer under the pooled fit than they are for the group they belong to.

Neither column is wrong arithmetic. They answer different questions — *how unusual among all
penguins* and *how unusual for a female Adelie* — and only one of those questions is about a
population that exists. "Three standard deviations from the mean" always carries a silent *of
which distribution*; here the pooled one was a fiction, and every z-score computed from it
was a statement about that fiction. The family was fine. The grouping was wrong.

What to *do* with a flagged bird is the part the numbers cannot settle, and it is the same
list as in 04.2: a mis-keyed measurement, a bird whose sex was recorded wrongly, or a real
large female. The tail probability tells you which birds to ask about; someone who handled
the birds tells you the answer.

## 4.3.5 Your turn: pooled versus per person

Your chat is a mixture in exactly this sense: every "messages per day" or "seconds between
messages" you compute over the whole export pools people with different habits. The move is
the one this notebook made three times — fit, notice the fit is wrong for the population,
split, fit again — and the question to answer is whether the split changes the *family* or
only the *parameters*.

Two candidates, either is enough:

- **Messages per day, pooled versus per author.** Fit the discrete families to the daily count
  over the whole chat, then to each author's daily count separately. `nbinom` will win pooled
  (no chat keeps a steady rate) — the question is whether the *dispersion* shrinks per person,
  or whether every person is as erratic as the group.
- **Gaps within a burst, pooled versus per author.** Cut the gaps at an hour as 04.4 does, fit
  `exponential` pooled and per author, and compare the *scale*. A pooled mean gap of ninety
  seconds can be one person at thirty and another at three hundred.

The recipe, in the calls this notebook used:

```python
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
per_day = own.set_index("timestamp").resample("D").size().to_numpy().astype(float)
pooled = fit_table(fitter.fit(per_day, discrete=True))

for author, messages in own.groupby("author"):
    per_day = messages.set_index("timestamp").resample("D").size().to_numpy().astype(float)
    describe_fit(author, per_day)  # adapt: discrete=True, and read nbinom's dispersion
```

Write down the pooled fit and the per-person fits as sentences with parameters in them, and
one sentence on whether the pooled number described anyone. Then take a message or a day that
the pooled fit flags as unusual and score it under its sender's own fit, the way §4.3.4 scored
the 202 mm Adelie.

## Reflection

1. Run the pooled fit on `body_mass_g` instead. The histogram looks like a long tail, mean sits
   above median, and the fitter is content with a skewed family — everything 04.2 taught you to
   read as a product mechanism. Split it the way this notebook split flipper length and say
   what the "tail" actually was.
2. The pooled Gentoo fit passed every test and was two groups. What has to be true of two
   groups for a mixture to hide like that, and what would you have to *know* — not compute — to
   suspect it?
3. For your own chat: name one variable where a fit over everyone would flag a message, a day,
   or a gap that is perfectly ordinary for the person it belongs to.

---

**Where this goes next.** Splitting by a group you already know about is one half of the
comparison move. The other half is splitting by *time* — the same family on both sides of an
event, and a null distribution to say whether the difference could be luck. That is
[04.4-before-and-after](04.4-before-and-after.ipynb), on the IRC showcase and then on your
own chat.